<div align="center">

# Remove near-duplicate texts with SemHash and fse

Find texts that say the same thing, keep one of each, and use fse as the encoder.

</div>

## When do you need this?

Text collections hold copies. A news feed repeats a story with a new headline. A training set repeats rows that also sit in the test set.
Exact matching finds only identical strings. **Semantic deduplication** finds texts whose vectors are very close.

[SemHash](https://github.com/MinishLab/semhash) does this. It encodes each text, searches for close neighbors, and keeps one text per group.
SemHash accepts any object with an `encode` method that turns a list of texts into a 2-D array. An fse `Encoder` has that method, so it plugs in without glue code.

| Step | What happens | Call |
|---|---|---|
| 1 | Load a pinned static model into an fse encoder | `fse.Encoder.from_pretrained` |
| 2 | Hand the encoder to SemHash | `SemHash.from_records(..., model=encoder)` |
| 3 | Remove near-duplicates inside one collection | `self_deduplicate` |
| 4 | Raise the similarity threshold without encoding again | `rethreshold` |
| 5 | Check new texts against a collection you already keep | `deduplicate` |
| 6 | Compare with SemHash's own Model2Vec encoder | `model2vec.StaticModel` |

**This recipe is a correctness check, not a benchmark.** It makes no speed claim and no claim about dedup quality.
Supported versions: **SemHash 0.5.0**. SemHash itself supports Python 3.10 and newer, but fse needs **Python 3.11 or newer**, so this recipe runs on 3.11 or newer only.
The [shared setup](README.md#run-a-recipe) installs the pinned packages, including `semhash` from `requirements.txt`.

## 1. Texts and an fse encoder

We use the 7,600 articles of the AG News test split, pinned in [_helpers.py](_helpers.py).
The model is [POTION-base-8M](https://huggingface.co/minishlab/potion-base-8M) at a pinned revision. It is English-only.
`recipe.encoder()` loads it with `fse.Encoder.from_pretrained`, with the settings stored in the model (including unit-length vectors).

In [1]:
from _helpers import Recipe
import fse
import numpy as np
from semhash import SemHash

recipe = Recipe()
texts, _ = recipe.dataset("AG News", "test")
encoder = recipe.encoder()
vectors = encoder.encode(texts[:3])
print("Texts:", len(texts))
print("encode returns:", type(vectors).__name__, vectors.dtype, vectors.shape)

Recipe run: 2026-11-15. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


/Users/oliverborchers/.herdr/worktrees/Fast_Sentence_Embeddings/dr428-semhash-recipe/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Texts: 7600
encode returns: ndarray float32 (3, 256)


## 2. Remove near-duplicates

SemHash calls `encoder.encode(list_of_texts)` once per column, with no other argument. That is all it asks of the encoder.
SemHash scales every vector to unit length itself, so it does not matter whether the encoder does.

By default SemHash searches neighbors with USearch, an **approximate** index, and a run can differ by a row or two from another.
We pass `ann_backend="basic"`, which compares every pair exactly, so the numbers below are repeatable and differ only when the encoder differs.
`threshold=0.9` means: a text is a duplicate when its cosine similarity to a text we keep is at least 0.9.

In [2]:
semhash = SemHash.from_records(texts, model=encoder, ann_backend="basic")
result = semhash.self_deduplicate(threshold=0.9)
print("Kept:", len(result.selected), "Removed:", len(result.filtered))
print("Exact string duplicates among the removed:", round(result.exact_duplicate_ratio, 4))

Kept: 7522 Removed: 78
Exact string duplicates among the removed: 0.0


Each removed text knows which kept text it duplicates and how similar they are. Read a few before you trust a threshold.
Texts are cut to 150 characters here.

In [3]:
def short(text, size=150):
    return " ".join(text.split())[:size]


for item in result.filtered[:3]:
    print(f"score {item.score:.3f}")
    print("  removed:", short(item.record))
    print("  kept:   ", short(item.duplicate_of))

score 0.989
  removed: Medtronic Quarterly Net Up CHICAGO (Reuters) - Medtronic Inc. &lt;A HREF="http://www.investor.reuters.com/FullQuote.aspx?ticker=MDT.N target=/stocks/q
  kept:    Medtronic Quarterly Earnings Rise CHICAGO (Reuters) - Medtronic Inc. &lt;A HREF="http://www.investor.reuters.com/FullQuote.aspx?ticker=MDT.N target=/s
score 0.974
  removed: Carly Patterson Wins Gymnastics All-Around Gold ATHENS (Reuters) - Carly Patterson upstaged Russian diva Svetlana Khorkina to become the first America
  kept:    Carly Patterson Wins the Women's All-Round ATHENS (Reuters) - Carly Patterson upstaged Russian diva Svetlana Khorkina to become the first American in 
score 0.951
  removed: Three people killed at Afghan checkpoint A man and two women were shot dead by Afghan and US-led troops after their vehicle ran through a checkpoint o
  kept:    Three People Killed at Afghan Checkpoint KABUL (Reuters) - A man and two women were shot dead by Afghan and U.S.-led troops after their vehicle

## 3. Move the threshold without encoding again

SemHash keeps the neighbor scores in the result. `result.rethreshold(value)` reuses them, so a different cutoff costs no encoder run and no new search.
It changes the result in place and can only **raise** the threshold, because pairs below the first threshold were never recorded.
So we start from a loose 0.8 and move up. A higher threshold removes fewer texts.
Choose the cutoff by reading removed pairs near it, as above, not by the count alone.

In [4]:
sweep = semhash.self_deduplicate(threshold=0.8)
print(f"threshold {sweep.threshold}: kept {len(sweep.selected)}, removed {len(sweep.filtered)}")
for threshold in (0.9, 0.95, 0.99):
    sweep.rethreshold(threshold)
    print(f"threshold {sweep.threshold}: kept {len(sweep.selected)}, removed {len(sweep.filtered)}")

threshold 0.8: kept 7251, removed 349
threshold 0.9: kept 7522, removed 78
threshold 0.95: kept 7573, removed 27


threshold 0.99: kept 7595, removed 5


## 4. Check new texts against a collection you keep

A common job is leakage control: which rows of a new set already exist in the set you hold?
Build the index on the first 5,000 texts, then pass the remaining 2,600 to `deduplicate`.
SemHash returns the new texts that are **not** near-duplicates of the reference.

In [5]:
reference = SemHash.from_records(texts[:5000], model=encoder, ann_backend="basic")
new_texts = texts[5000:]
checked = reference.deduplicate(new_texts, threshold=0.9)
print("New texts:", len(new_texts), "New to the reference:", len(checked.selected), "Already known:", len(checked.filtered))
for item in checked.filtered[:2]:
    print(f"score {item.score:.3f}")
    print("  new text:", short(item.record))
    print("  known:   ", short(item.duplicate_of))

New texts: 2600 New to the reference: 2595 Already known: 5
score 0.936
  new text: Death toll climbs in Baghdad blast A car bomb exploded outside the education ministry in central Baghdad on Tuesday, killing at least six people and w
  known:    Car Bomb Kills at Least Six in Baghdad A car bomb exploded outside the Education Ministry in central Baghdad Tuesday, killing at least six people and 
score 0.946
  new text: Symantec adds threat data to Managed Security Services SAN FRANCISCO - In a bid to expand its services business, Symantec Corp. next week plans to sta
  known:    McData offers SAN consolidation McData plans to introduce a new SAN router this week designed to connect the growing number of isolated SAN networks i


Read the pairs. The first is the same story with a new headline. The second is **not** a duplicate: the Symantec and McData texts only share the term "SAN" (storage network or San Francisco).
A high score is a reason to look, not proof. The same holds in section 2: check the pairs at your threshold before you delete data.

## 5. Compare with SemHash's default encoder

Without `model=`, SemHash loads `minishlab/potion-base-8M` through Model2Vec from the Hub's current revision.
To compare the **encoders** and not the revisions, we load the same pinned files with `model2vec.StaticModel`.
Both are POTION-base-8M, so we expect near-identical vectors and the same decisions.
We report what we observe. Tokenization, truncation and float rounding can differ between two implementations.

In [6]:
from model2vec import StaticModel

default = StaticModel.from_pretrained(str(recipe.model_folder()))
ours, theirs = encoder.encode(texts), default.encode(texts)
print("Largest vector difference:", float(np.abs(ours - theirs).max()))

baseline = SemHash.from_records(texts, model=default, ann_backend="basic").self_deduplicate(threshold=0.9)
same_kept = set(result.selected) == set(baseline.selected)
print("Model2Vec encoder, kept:", len(baseline.selected), "removed:", len(baseline.filtered))
print("fse encoder,       kept:", len(result.selected), "removed:", len(result.filtered))
print("Same kept texts:", same_kept)

Largest vector difference: 5.960464477539063e-08


Model2Vec encoder, kept: 7522 removed: 78
fse encoder,       kept: 7522 removed: 78
Same kept texts: True


## Limits and next steps

- **Versions.** Tested with SemHash 0.5.0 and fse on Python 3.13. SemHash supports Python 3.10, fse needs 3.11 or newer.
  Another SemHash version may change the encoder contract or its defaults. This recipe checks no other version.
- **Model.** POTION-base-8M is a small English model. It sees topic and wording, not facts. Two articles on one event can look close, and a changed number can look identical.
  Use a model that fits your language and domain.
- **Normalization.** SemHash scales vectors itself, so fse's own normalization setting does not change the decisions here. Another SemHash version may differ.
- **What changes against the default encoder.** Only the object that turns texts into vectors. The index, the threshold and the grouping stay SemHash's. In this run both encoders gave the same result (see section 5), which is an observation on 7,600 texts, not a guarantee.
- **Search.** The exact backend compares every pair and grows quadratically with the collection. For larger collections use SemHash's default approximate search and expect small run-to-run differences.
- **No quality claim.** We did not label which pairs are true duplicates, so we report no precision or recall. No speed is measured.
- **No glue package.** This recipe uses public fse names only. There is no fse adapter for SemHash.

- [fse 101](00-fse-101.ipynb)
- [External indexes](10-external-ann-indexes.ipynb)
- [Recipe index](README.md)

In [7]:
recipe.close()